<a href="https://colab.research.google.com/github/amoghkamath06/AI_lab/blob/main/resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from itertools import count

# ---------------------------------------------------------------- helpers
def is_var(t):
    return isinstance(t, str) and t[0].islower()

def lit(s):
    """'~eats(x,y)' -> (True, 'eats', ('x','y'))"""
    s = s.strip()
    neg = s.startswith("~")
    if neg:
        s = s[1:]
    pred, args = s.split("(")
    args = tuple(a.strip() for a in args.rstrip(")").split(","))
    return (neg, pred, args)

def clause(s):
    """'~food(x) | likes(John,x)' -> frozenset of literals"""
    return frozenset(lit(p) for p in s.split("|"))

def show_lit(l):
    return ("~" if l[0] else "") + f"{l[1]}({','.join(l[2])})"

def show(c):
    return " | ".join(sorted(show_lit(l) for l in c)) if c else "[] (empty clause)"

# ---------------------------------------------------------------- unification
def subst(t, s):
    while is_var(t) and t in s:
        t = s[t]
    return t

def unify_terms(a, b, s):
    a, b = subst(a, s), subst(b, s)
    if a == b:
        return s
    if is_var(a):
        return {**s, a: b}
    if is_var(b):
        return {**s, b: a}
    return None          # two different constants

def unify(args1, args2):
    s = {}
    for a, b in zip(args1, args2):
        s = unify_terms(a, b, s)
        if s is None:
            return None
    return s

def apply(c, s):
    return frozenset((n, p, tuple(subst(a, s) for a in args)) for n, p, args in c)

_ids = count(1)
def rename(c):
    """Standardise variables apart."""
    k = next(_ids)
    return frozenset((n, p, tuple(f"{a}{k}" if is_var(a) else a for a in args))
                     for n, p, args in c)

def tautology(c):
    return any((not n, p, a) in c for n, p, a in c)

# ---------------------------------------------------------------- resolution
def resolvents(c1, c2):
    c1, c2 = rename(c1), rename(c2)
    for l1 in c1:
        for l2 in c2:
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                s = unify(l1[2], l2[2])
                if s is not None:
                    r = apply((c1 - {l1}) | (c2 - {l2}), s)
                    yield r, s

def canon(c):
    """Rename variables canonically so duplicates are recognised."""
    m, out = {}, []
    for n, p, args in sorted(c, key=show_lit):
        out.append((n, p, tuple(m.setdefault(a, f"v{len(m)}") if is_var(a) else a for a in args)))
    return frozenset(out)

def prove(kb, negated_goal):
    clauses = [(c, "KB") for c in kb] + [(negated_goal, "negated goal")]
    seen = {canon(c) for c, _ in clauses}
    parents = {}                       # index -> (i, j, unifier)
    sos = [len(clauses) - 1]           # set-of-support strategy: start from goal

    while sos:
        new_sos = []
        for i in sos:
            for j in range(len(clauses)):
                if j == i:
                    continue
                for r, s in resolvents(clauses[i][0], clauses[j][0]):
                    if tautology(r) or canon(r) in seen:
                        continue
                    seen.add(canon(r))
                    clauses.append((r, "resolvent"))
                    k = len(clauses) - 1
                    parents[k] = (i, j, s)
                    if not r:
                        return clauses, parents, k
                    new_sos.append(k)
        sos = new_sos
    return clauses, parents, None

# ---------------------------------------------------------------- output
def print_proof(clauses, parents, goal_idx, n_kb):
    used, stack = set(), [goal_idx]
    while stack:
        k = stack.pop()
        if k in used:
            continue
        used.add(k)
        if k in parents:
            stack += parents[k][:2]
    order = sorted(used)
    num = {k: n + 1 for n, k in enumerate(order)}

    print("\n" + "="*85)
    print(" "*28 + " RESOLUTION PROOF STEPS ")
    print("="*85)
    for k in order:
        c, why = clauses[k]
        if k in parents:
            i, j, s = parents[k]
            u = ", ".join(f"{a}/{b}" for a, b in s.items()) or "-"
            print(f" Step {num[k]:>2}:  {show(c):<45} [Resolve {num[i]} & {num[j]} with {{{u}}}]")
        else:
            print(f" Step {num[k]:>2}:  {show(c):<45} [{why}]")


    # Generate a hierarchical top-down tree representation
    print("\n" + "="*85)
    print(" "*24 + " RESOLUTION PROOF TREE ")
    print("="*85)

    def get_tree_structure(node_idx):
        c, why = clauses[node_idx]
        node_str = f"Step {num[node_idx]}: {show(c) if c else '[] (Empty Clause)'}"
        if node_idx not in parents:
            return {"text": f"{node_str} ({why})", "children": []}

        left_p, right_p, _ = parents[node_idx]
        return {
            "text": node_str,
            "children": [get_tree_structure(left_p), get_tree_structure(right_p)]
        }

    # Formulate and print top-down (leafs/KB at top, converging downwards)
    def print_top_down(tree_node, prefix="", is_last=True):
        # Print children (ancestors/sources) first to build top-down flow
        if tree_node["children"]:
            left, right = tree_node["children"]
            print_top_down(left, prefix + ("    " if is_last else "│   "), False)
            print_top_down(right, prefix + ("    " if is_last else "│   "), True)

        # Print current resolution/derived clause
        print(prefix + ("└── " if is_last else "├── ") + tree_node["text"])

    root_tree = get_tree_structure(goal_idx)
    print_top_down(root_tree)
    print("="*85)

# ---------------------------------------------------------------- case study
if __name__ == "__main__":
    print("""Knowledge base in FOL
  a. forall x  food(x) -> likes(John,x)
  b. food(Apple) ^ food(Vegetables)
  c. forall x,y (eats(x,y) ^ ~killed(x)) -> food(y)
  d. eats(Anil,Peanuts) ^ alive(Anil)
  e. forall x  eats(Anil,x) -> eats(Harry,x)
  f. forall x  alive(x) -> ~killed(x)
  g. forall x  ~killed(x) -> alive(x)
  Query: likes(John,Peanuts)""")

    kb = [
        clause("~food(x) | likes(John,x)"),
        clause("food(Apple)"),
        clause("food(Vegetables)"),
        clause("~eats(x,y) | killed(x) | food(y)"),
        clause("eats(Anil,Peanuts)"),
        clause("alive(Anil)"),
        clause("~eats(Anil,x) | eats(Harry,x)"),
        clause("~alive(x) | ~killed(x)"),
        clause("killed(x) | alive(x)"),
    ]
    print("\nCNF clauses")
    for n, c in enumerate(kb, 1):
        print(f"  {n}. {show(c)}")

    goal = clause("likes(John,Peanuts)")
    neg_goal = frozenset({(not l[0], l[1], l[2]) for l in goal})
    print("\nNegated goal:", show(neg_goal))

    clauses, parents, found = prove(kb, neg_goal)
    if found is None:
        print("\nResolution exhausted: query NOT provable.")
    else:
        print_proof(clauses, parents, found, len(kb) + 1)
        print("\nEmpty clause derived -> negated goal is contradictory"\
              "\n=> John likes peanuts.")